## Lecture du bronze

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

BRONZE = "abfss://bronze@stbixipulse2611.dfs.core.windows.net"
TARGET = "bixi.silver.station_status"

raw = spark.read.parquet(f"{BRONZE}/station_status")
print(f"Bronze: {raw.count() :,} lignes")

## Le dédoublonnage entre les 2 collecteurs

In [0]:
latest_first = Window.partitionBy("station_id", "source_last_updated").orderBy(
    F.col("ingested_at").desc()
)

deduped = (
    raw.withColumn("rn", F.row_number().over(latest_first))
    .filter(F.col("rn") == 1)
    .drop("rn")
)
n_raw, n_dedup = raw.count(), deduped.count()
print(f"Après dédoublonnage : {n_dedup:,} lignes ({n_raw - n_dedup:,} doublons supprimés)")

## Un schéma propre

In [0]:
silver = deduped.select(
    "station_id",
    F.col("source_last_updated").alias("snapshot_at"),
    F.timestamp_seconds("last_reported").alias("last_reported_at"),
    F.col("num_bikes_available").cast("int"),
    F.col("num_ebikes_available").cast("int"),
    F.col("num_bikes_disabled").cast("int"),
    F.col("num_docks_available").cast("int"),
    F.col("num_docks_disabled").cast("int"),
    (F.col("is_installed") == 1).alias("is_installed"),
    (F.col("is_renting") == 1).alias("is_renting"),
    (F.col("is_returning") == 1).alias("is_returning"),
    "ingested_at",
)
silver.printSchema()

## Les contrôles qualité

In [0]:
checks = {
    "station_id manquant": silver.filter(F.col("station_id").isNull()).count(),
    "vélos disponibles négatifs": silver.filter(F.col("num_bikes_available") < 0).count(),
    "bornes disponibles négatives": silver.filter(F.col("num_docks_available") < 0).count(),
    "snapshot dans le futur": silver.filter(F.col("snapshot_at") > F.current_timestamp()).count(),
}

for name, n in checks.items():
    print(f"{'✅' if n == 0 else '❌'} {name} : {n}")

failed = {name: n for name, n in checks.items() if n > 0}
assert not failed, f"Contrôles qualité en échec : {failed}"

## L’écriture, puis une vérification

In [0]:
(
    silver.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TARGET)
)

display(
    spark.table(TARGET)
    .groupBy("is_installed", "is_renting")
    .count()
    .orderBy("is_installed", "is_renting")
)